In [ ]:
# struct_parser.py — mmap shared-region + netlink helpers + row conversion
import os, mmap, struct, socket
from typing import Dict, Any, List, Tuple, Optional
from config import (
    DEV_PATH, REG_SIZE, FEAT_FMT, WIN_HDR_FMT, REG_HDR_FMT,
    MAX_PROCS, SEQ_LEN, NETLINK_PROTOCOL
)
from utils import pfn_to_slices, va_to_L1L2, latency_to_cluster

FEAT_SIZE = struct.calcsize(FEAT_FMT)
WIN_HDR_SIZE = struct.calcsize(WIN_HDR_FMT)
REG_HDR_SIZE = struct.calcsize(REG_HDR_FMT)
WIN_SIZE = WIN_HDR_SIZE + SEQ_LEN * FEAT_SIZE

# --- mmap/open
def open_region():
    """Safely try to open and mmap /dev/swap_ai; fall back if unavailable."""
    if not os.path.exists(DEV_PATH):
        print("⚠ WARNING: /dev/swap_ai not found — kernel tracking unavailable.")
        print("⚠ Daemon will run in MODEL-ONLY MODE.")
        return None, None

    try:
        fd = os.open(DEV_PATH, os.O_RDONLY)
    except Exception as e:
        print(f"⚠ ERROR: Cannot open {DEV_PATH}: {e}")
        print("⚠ Running in MODEL-ONLY MODE.")
        return None, None

    try:
        mm = mmap.mmap(fd, REG_SIZE, access=mmap.ACCESS_READ)
    except Exception as e:
        print(f"⚠ ERROR: mmap failed: {e}")
        print("⚠ Running in MODEL-ONLY MODE.")
        try:
            os.close(fd)
        except:
            pass
        return None, None

    print(f"✅ Kernel shared-region mapped: {REG_SIZE} bytes")
    return fd, mm

# --- netlink
def open_netlink():
    nl = socket.socket(socket.AF_NETLINK, socket.SOCK_RAW, NETLINK_PROTOCOL)
    nl.bind((os.getpid(), 0))
    nl.send(b"HELLO")
    return nl

NLMSG_HDR_FMT = "<IHHII"   # len,type,flags,seq,pid
NLMSG_HDR_SIZE = struct.calcsize(NLMSG_HDR_FMT)

def nl_get_payload_pid(dat: bytes) -> Optional[int]:
    if len(dat) < NLMSG_HDR_SIZE + 4:
        return None
    _,_,_,_,_ = struct.unpack_from(NLMSG_HDR_FMT, dat, 0)
    payload_off = NLMSG_HDR_SIZE
    if payload_off + 4 <= len(dat):
        (pid_payload,) = struct.unpack_from("<I", dat, payload_off)
        return pid_payload
    return None

# --- region reading
def read_region(mm) -> List[Dict[str,Any]]:
    if not mm:
        print("Kernel mm not found..")
        return None  
    nslots, seqlen = struct.unpack_from(REG_HDR_FMT, mm, 0)
    windows = []
    base = REG_HDR_SIZE
    for i in range(MAX_PROCS):
        off = base + i * WIN_SIZE
        pid, count, comm = struct.unpack_from(WIN_HDR_FMT, mm, off)
        comm = comm.split(b"\x00",1)[0].decode(errors="ignore")
        feats = []
        feat_off = off + WIN_HDR_SIZE
        for j in range(SEQ_LEN):
            va,pfn,mapping,start_ns,latency_ns,folio_idx,_ = struct.unpack_from(FEAT_FMT, mm, feat_off + j*FEAT_SIZE)
            feats.append((va,pfn,mapping,start_ns,latency_ns,folio_idx))
        windows.append({"pid":pid,"count":count,"comm":comm,"feats":feats})
    return windows

def find_pid_window(windows, pid) -> Optional[Dict[str,Any]]:
    for w in windows:
        if w["pid"] == pid:
            return w
    return None

# --- convert one feat tuple → V3 row
def feat_to_row(pid:int, feat_tuple) -> Dict[str,Any]:
    va, pfn, mapping, start_ns, latency_ns, folio_idx = feat_tuple
    top,s4,s3,s2,s1,s0 = pfn_to_slices(int(pfn))
    va_l2, va_l1 = va_to_L1L2(int(va))
    return {
        'Va_L2': va_l2,
        'Va_L1': va_l1,
        'PFN_Top_region': top,
        'PFN_slice_4': s4,
        'PFN_slice_3': s3,
        'PFN_slice_2': s2,
        'PFN_slice_1': s1,
        'PFN_slice_0': s0,
        'PID': int(pid),
        'folio_index': int(folio_idx),
        'mapping': int(mapping),
        'lat_cluster': latency_to_cluster(int(latency_ns)),
        'start_ns': int(start_ns),
        # PFN will be reconstructed by queue if needed (not strictly required here)
    }


In [ ]:
import os 

In [ ]:
os.open('daemon_main.py',os.O_RDONLY)
mmap.map()

3

In [3]:
os.getpid()

1956